# Experiment: In-Process Pipeline Execution

Run the KG construction pipeline **directly in-process** — no server required.

Unlike `batch_processing.ipynb` (which calls the server via HTTP), this notebook loads the pipeline config and SPG schema, then executes `PipelineWorkflow` directly. This gives you:
- Direct access to pipeline state and intermediate results
- Ability to override config values inline before running
- Faster iteration for experimentation

## Prerequisites

1. Environment variables set (`.env` file with NEO4J credentials, API keys)
2. Input data files available (e.g. `data/financebench/`)
3. Dependencies installed: `uv sync`

## 1. Setup and Imports

In [1]:
import sys
import os
import json
import logging
from pathlib import Path
from pprint import pprint
from datetime import datetime

# Add project root to path so we can import packages directly
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables from .env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Project imports
from knowledge_graphs.pipeline.config import load_config, resolve_environment_variables
from knowledge_graphs.pipeline.langgraph_executor import PipelineWorkflow
from knowledge_graphs.models.schema import DomainSchema

# Configure logging to see pipeline progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(name)s] %(levelname)s: %(message)s")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python path configured")

Project root: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph
Python path configured


## 2. Load Pipeline Config

Load the YAML config and resolve `${VAR}` environment variable references.

In [2]:
# Path to the pipeline config YAML (relative to project root)
CONFIG_PATH = PROJECT_ROOT / "configs" / "financebench_pipeline.yaml"

# Load and resolve environment variables
config = load_config(CONFIG_PATH)
config = resolve_environment_variables(config)

# Display pipeline overview
pipeline = config["pipeline"]
print(f"Pipeline: {pipeline['name']}")
print(f"Version:  {pipeline['version']}")
print(f"Workers:  {pipeline['max_workers']}")
print(f"Timeout:  {pipeline.get('timeout', 'None')}s")

print(f"\nComponents:")
for name, comp in pipeline["components"].items():
    status = "ON" if comp.get("enabled", False) else "OFF"
    print(f"  [{status:>3}] {name}: {comp['type']}")

2026-02-25 14:51:08,348 [knowledge_graphs.pipeline.config] INFO: Loaded configuration from /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph/configs/financebench_pipeline.yaml


Pipeline: financebench_extraction_pipeline
Version:  1.0
Workers:  4
Timeout:  3600s

Components:
  [ ON] scanner: file_scanner
  [ ON] reader: financebench_reader
  [OFF] splitter: semantic_splitter
  [ ON] extractor: llm_extractor
  [ ON] vectorizer: gemini_vectorizer
  [ ON] writer: neo4j_writer


## 3. Load & Inspect Schema

Load the SPG domain schema referenced in the extractor config and inspect its entity/relation types.

In [3]:
# Resolve schema path from extractor config
extractor_config = pipeline["components"]["extractor"]["config"]
schema_rel_path = extractor_config.get("extraction_schema", "")
schema_path = PROJECT_ROOT / schema_rel_path

print(f"Schema file: {schema_path}")
print(f"Exists: {schema_path.exists()}\n")

# Load and inspect
schema = DomainSchema.from_file(str(schema_path))

print(f"Namespace: {schema.namespace}")
print(f"\nEntity Types ({len(schema.entity_type_names)}):")
for name in schema.entity_type_names:
    edef = schema.entities[name]
    n_props = len(edef.properties)
    n_rels = len(edef.relations)
    print(f"  {edef.entity_type:<12} {name:<25} ({n_props} props, {n_rels} relations)")

print(f"\nRelation Types ({len(schema.relation_type_names)}):")
for rel_name in schema.relation_type_names:
    print(f"  {rel_name}")

2026-02-25 14:51:33,376 [knowledge_graphs.models.schema] WARNING: Entity 'RegulatoryFiling', relation 'COVERS_PERIOD': target_entity 'Text' is not defined in the schema


Schema file: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph/knowledge_graphs/schema/financebench_spg.yaml
Exists: True

Namespace: Finance

Entity Types (15):
  EntityType   BusinessSegment           (3 props, 2 relations)
  EntityType   Chunk                     (3 props, 0 relations)
  EntityType   Company                   (6 props, 10 relations)
  ConceptType  EventCategory             (0 props, 1 relations)
  EntityType   Executive                 (7 props, 2 relations)
  ConceptType  ExecutiveRole             (0 props, 1 relations)
  EntityType   FinancialEvent            (6 props, 2 relations)
  EntityType   FinancialStatement        (6 props, 2 relations)
  EntityType   GeographicRegion          (6 props, 1 relations)
  ConceptType  Industry                  (0 props, 1 relations)
  EntityType   MarketData                (7 props, 0 relations)
  EntityType   Product                   (5 props, 2 relations)
  ConceptType  ProductCategory           (0 props, 1 relations

## 4. Config Overrides

Tweak config values before running the pipeline. Uncomment and modify as needed.

In [4]:
# --- Scanner overrides ---
# config["pipeline"]["components"]["scanner"]["config"]["max_files"] = 1
# config["pipeline"]["components"]["scanner"]["config"]["input_paths"] = ["./data/financebench"]

# --- Extractor overrides ---
# config["pipeline"]["components"]["extractor"]["config"]["model"] = "gpt-4o-mini"
# config["pipeline"]["components"]["extractor"]["config"]["temperature"] = 0.0
# config["pipeline"]["components"]["extractor"]["config"]["batch_size"] = 3

# --- Writer overrides ---
# config["pipeline"]["components"]["writer"]["config"]["database"] = "experiment_01"
# config["pipeline"]["components"]["writer"]["config"]["clear_database"] = True

# --- Enable/disable components ---
# config["pipeline"]["components"]["splitter"]["enabled"] = True
# config["pipeline"]["components"]["vectorizer"]["enabled"] = False

print("Config overrides applied (edit this cell to customize)")
print(f"  Scanner max_files: {config['pipeline']['components']['scanner']['config'].get('max_files', 'unlimited')}")
print(f"  Extractor model:   {config['pipeline']['components']['extractor']['config']['model']}")
print(f"  Writer database:   {config['pipeline']['components']['writer']['config']['database']}")

Config overrides applied (edit this cell to customize)
  Scanner max_files: 1
  Extractor model:   gemini-2.5-flash
  Writer database:   financebench1


## 5. Create & Run Pipeline

Create `PipelineWorkflow` directly with the config dict and run it in-process.

In [5]:
# Input path for the pipeline (resolved relative to project root)
INPUT_PATH = str(PROJECT_ROOT / "data" / "financebench")

# Create workflow (this initializes all components from config)
workflow = PipelineWorkflow(config=config)

# Display pipeline info
info = workflow.get_pipeline_info()
print(f"Workflow ready with {info['workflow_nodes']} components:")
for name, comp_info in info["components"].items():
    print(f"  {name}: {comp_info['class']} (enabled={comp_info['enabled']})")

2026-02-25 14:52:24,942 [knowledge_graphs.pipeline.langgraph_executor] INFO: Initialized scanner component: file_scanner
2026-02-25 14:52:24,943 [knowledge_graphs.pipeline.langgraph_executor] INFO: Initialized reader component: financebench_reader
2026-02-25 14:52:25,015 [sentence_transformers.SentenceTransformer] INFO: Use pytorch device_name: mps
2026-02-25 14:52:25,016 [sentence_transformers.SentenceTransformer] INFO: Load pretrained SentenceTransformer: jhu-clsp/mmBERT-small
2026-02-25 14:52:25,512 [sentence_transformers.SentenceTransformer] WARNING: No sentence-transformers model found with name jhu-clsp/mmBERT-small. Creating a new one with mean pooling.
2026-02-25 14:52:30,934 [knowledge_graphs.components.splitter] INFO: Initialized SentenceTransformer with model: jhu-clsp/mmBERT-small
2026-02-25 14:52:30,937 [knowledge_graphs.pipeline.langgraph_executor] INFO: Initialized splitter component: semantic_splitter
2026-02-25 14:52:30,986 [knowledge_graphs.components.extractor] ERROR

Workflow ready with 6 components:
  scanner: FileScanner (enabled=True)
  reader: FinanceBenchReader (enabled=True)
  splitter: SemanticSplitter (enabled=True)
  extractor: LLMExtractor (enabled=True)
  vectorizer: GeminiVectorizer (enabled=True)
  writer: Neo4jWriter (enabled=True)


In [6]:
# Run the pipeline
print(f"Starting pipeline at {datetime.now().strftime('%H:%M:%S')}")
print(f"Input: {INPUT_PATH}")
print("=" * 60)

start_time = datetime.now()
results = workflow.run_pipeline(INPUT_PATH)
elapsed = (datetime.now() - start_time).total_seconds()

print("=" * 60)
print(f"Pipeline finished in {elapsed:.1f}s")
print(f"Status: {results['status']}")

if results["errors"]:
    print(f"\nErrors ({len(results['errors'])}):")
    for err in results["errors"]:
        print(f"  - {err}")

2026-02-25 14:52:39,468 [knowledge_graphs.pipeline.langgraph_executor] INFO: Starting pipeline run 3df99c6e-9094-4173-b141-dbe7590b3c1a for input: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph/data/financebench
2026-02-25 14:52:39,508 [knowledge_graphs.pipeline.langgraph_executor] INFO: Executing component: scanner
2026-02-25 14:52:39,509 [knowledge_graphs.pipeline.langgraph_executor] ERROR: Component scanner failed after 0.00s: File not found: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph/data/financebench
2026-02-25 14:52:39,511 [knowledge_graphs.pipeline.langgraph_executor] INFO: Executing component: reader
2026-02-25 14:52:39,511 [knowledge_graphs.pipeline.langgraph_executor] ERROR: Input validation failed for 'reader': Reader requires 'file_paths' from Scanner. Upstream component 'scanner' likely did not produce the required data.
2026-02-25 14:52:39,512 [knowledge_graphs.pipeline.langgraph_executor] INFO: Executing component: splitter
2026-02-25 14:52:39,513

Starting pipeline at 14:52:39
Input: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph/data/financebench
Pipeline finished in 0.0s
Status: PipelineStatus.COMPLETED

Errors (6):
  - scanner: File not found: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph/data/financebench
  - reader: Input validation failed for 'reader': Reader requires 'file_paths' from Scanner. Upstream component 'scanner' likely did not produce the required data.
  - splitter: Input validation failed for 'splitter': Splitter requires 'chunks' from Reader. Upstream component 'reader' likely did not produce the required data.
  - extractor: Input validation failed for 'extractor': Extractor requires 'split_chunks' or 'chunks'. Upstream component 'splitter (or reader)' likely did not produce the required data.
  - vectorizer: Input validation failed for 'vectorizer': Vectorizer requires 'subgraphs' from Extractor. Upstream component 'extractor' likely did not produce the required data.
  - writer: Input 

## 6. Inspect Results

View metrics, per-component timing, and execution summary.

In [ ]:
# Metrics summary
metrics = results.get("metrics", {})
if metrics:
    # Handle both dict and Pydantic model
    m = metrics if isinstance(metrics, dict) else metrics.dict() if hasattr(metrics, "dict") else vars(metrics)

    print("Metrics Summary")
    print("-" * 40)
    print(f"  Files processed:    {m.get('total_files_processed', 'N/A')}")
    print(f"  Chunks created:     {m.get('total_chunks_created', 'N/A')}")
    print(f"  Nodes extracted:    {m.get('total_nodes_extracted', 'N/A')}")
    print(f"  Edges extracted:    {m.get('total_edges_extracted', 'N/A')}")
    print(f"  Subgraphs created:  {m.get('total_subgraphs_created', 'N/A')}")
    print(f"  Total time:         {m.get('total_execution_time', 'N/A')}s")

    # Per-component timing
    component_times = m.get("component_times", {})
    if component_times:
        print(f"\nPer-Component Timing")
        print("-" * 40)
        for comp, t in component_times.items():
            print(f"  {comp:<15} {t:.2f}s")
else:
    print("No metrics available")

# Execution summary
summary = results.get("execution_summary")
if summary:
    print(f"\nExecution Summary")
    print("-" * 40)
    pprint(summary)

## 7. Verify Neo4j (Optional)

Connect to Neo4j and check what was written. Requires Neo4j to be running.

In [ ]:
try:
    from neo4j import GraphDatabase

    NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
    NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
    NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
    NEO4J_DATABASE = config["pipeline"]["components"]["writer"]["config"]["database"]

    if not NEO4J_PASSWORD:
        print("NEO4J_PASSWORD not set, skipping database check")
    else:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

        with driver.session(database=NEO4J_DATABASE) as session:
            # Count nodes and relationships
            node_count = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
            rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]

            # Get node label distribution
            labels = session.run(
                "MATCH (n) UNWIND labels(n) AS label "
                "RETURN label, count(*) AS count ORDER BY count DESC LIMIT 10"
            ).data()

            # Sample entities
            samples = session.run(
                "MATCH (n) WHERE n.name IS NOT NULL "
                "RETURN labels(n)[0] AS label, n.name AS name LIMIT 10"
            ).data()

        driver.close()

        print(f"Neo4j Database: {NEO4J_DATABASE}")
        print(f"  Total nodes:         {node_count}")
        print(f"  Total relationships: {rel_count}")

        if labels:
            print(f"\n  Node Labels:")
            for row in labels:
                print(f"    {row['label']:<25} {row['count']}")

        if samples:
            print(f"\n  Sample Entities:")
            for row in samples:
                print(f"    [{row['label']}] {row['name']}")

except ImportError:
    print("neo4j driver not installed. Install with: uv add neo4j")
except Exception as e:
    print(f"Could not connect to Neo4j: {e}")

## 8. Next Steps

Ideas for further experiments:

- **Change `max_files`** in scanner config to process more/fewer files
- **Switch extractor model** (e.g. `gpt-4o` vs `gemini-2.5-flash`) and compare extraction quality
- **Enable the splitter** to test semantic chunking vs page-based chunking
- **Use `workflow.stream_pipeline()`** to watch real-time progress per component
- **Use `workflow.dry_run()`** to validate config without executing:

```python
validation = workflow.dry_run(INPUT_PATH)
pprint(validation)
```

- **Resume from checkpoint** after a failure:

```python
results = workflow.resume_pipeline(pipeline_id="<id-from-results>")
```